# Fine-tuning Llama-3 para Assistente Médico (Fase 3) - Dados Reais

Este notebook realiza o fine-tuning de um modelo Llama-3 usando datasets reais de Saúde da Mulher (MedQuAD e Women Health).

In [ ]:
!pip install -q -U transformers peft bitsandbytes datasets accelerate trl

In [ ]:
from datasets import load_dataset
import json
import os
import shutil

def prepare_real_data():
    # Se subir o arquivo gerado localmente ou na pasta data, da pra usar ele
    if os.path.exists("real_medical_data.jsonl"):
        print("Dataset real já existe localmente.")
        return
    elif os.path.exists("../data/real_medical_data.jsonl"):
        print("Copiando dataset real da pasta data...")
        shutil.copy("../data/real_medical_data.jsonl", "real_medical_data.jsonl")
        return

    print("Baixando datasets reais do Hugging Face (MedQuAD e Women Health)...")
    processed_data = []
    
    # 1. Carregar Women Health
    try:
        ds_women = load_dataset("altaidevorg/women-health-mini", split="train")
        for item in ds_women:
            conversations = item.get("conversations", [])
            instruction = ""
            response = ""
            for msg in conversations:
                if msg.get("role") == "user":
                    instruction = msg.get("content", "")
                elif msg.get("role") == "assistant":
                    response = msg.get("content", "")
            if instruction and response:
                processed_data.append({
                    "instruction": instruction,
                    "context": "Saúde da Mulher / Ginecologia",
                    "response": response
                })
        print(f"Women Health carregado: {len(processed_data)} exemplos.")
    except Exception as e:
        print(f"Erro ao carregar Women Health: {e}")

    # 2. Carregar MedQuAD (Subset filtrado)
    try:
        ds_medquad = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")
        keywords = ["woman", "women", "pregnancy", "breast", "gynecology", "obstetrics", "menstrual", "urology"]
        ds_medquad_filtered = ds_medquad.filter(lambda x: any(k in x['Question'].lower() for k in keywords))
        for item in ds_medquad_filtered:
            processed_data.append({
                "instruction": item.get("Question", ""),
                "context": "Saúde da Mulher / Triagem Médica",
                "response": item.get("Answer", "")
            })
        print(f"MedQuAD filtrado adicionado. Total: {len(processed_data)} exemplos.")
    except Exception as e:
        print(f"Erro ao carregar MedQuAD: {e}")

    if processed_data:
        with open("real_medical_data.jsonl", "w", encoding="utf-8") as f:
            for entry in processed_data:
                f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        print(f"Dataset real preparado com sucesso: real_medical_data.jsonl ({len(processed_data)} exemplos)")
    else:
        print("Nenhum dado pôde ser preparado.")

prepare_real_data()

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

model_id = "unsloth/llama-3-8b-bnb-4bit"
dataset_path = "real_medical_data.jsonl"

dataset = load_dataset("json", data_files=dataset_path, split="train")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

training_args = SFTConfig(
    output_dir="./llama3-medical-fase3",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=100,
    logging_steps=10,
    fp16=False,
    bf16=False,
    dataset_text_field="instruction",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args
)

trainer.train()
print("Treinamento Concluído!")